# 40 — Report-ready views from actual runs
This results template reads only canonical outputs/<run_id>/tables files. Set RUN_ID to a completed run. Empty inputs produce empty tables and an explicit message; no numbers are fabricated.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
RUN_ID = os.getenv('RUN_ID', '').strip()
OUTPUT_ROOT = Path(os.getenv('OUTPUT_ROOT', str(ROOT / 'outputs'))).resolve()
assert OUTPUT_ROOT.is_relative_to(ROOT), 'OUTPUT_ROOT must stay inside the repository'
OUT = OUTPUT_ROOT / RUN_ID / 'tables' if RUN_ID else None
def read_actual(name, columns):
    paths = sorted(OUT.glob(f'**/{name}')) if OUT and OUT.is_dir() else []
    return pd.concat([pd.read_csv(p) for p in paths], ignore_index=True) if paths else pd.DataFrame(columns=columns)
metrics = read_actual('metrics.csv', ['model', 'seed', 'fold', 'cost_bps', 'Sharpe Ratio', 'Annualized Turnover', 'train_seconds', 'parameters'])
regimes = read_actual('regimes.csv', ['model', 'seed', 'fold', 'regime', 'sharpe', 'sortino', 'max_drawdown', 'annual_return'])
returns = read_actual('returns.csv', ['model', 'seed', 'fold', 'decision_date', 'target_date', 'gross_return', 'turnover'])
predictions = read_actual('predictions.csv', ['model', 'seed', 'fold', 'asset', 'signal', 'target_return'])
if metrics.empty: print('No canonical benchmark outputs found. Set RUN_ID to a completed run.')

## Primary economic metrics and cost sensitivity
Keep each seed visible. Net Sharpe is calculated independently for each fee level. Treat architecture rank as descriptive until uncertainty and multiple comparisons are addressed.

In [ ]:
if not metrics.empty:
    display(metrics.sort_values(['cost_bps', 'model', 'seed']))
    cost_view = metrics.groupby(['model', 'cost_bps'])['Sharpe Ratio'].agg(['mean', 'std']).reset_index()
    display(cost_view)
    ax = metrics.groupby(['cost_bps', 'model'])['Sharpe Ratio'].mean().unstack().plot(marker='o', figsize=(8, 4), title='Net Sharpe vs transaction costs')
    ax.set_ylabel('Annualized Sharpe'); ax.set_xlabel('Basis points'); plt.show()

## Regime robustness
Display regime Sharpe, worst regime, and best-minus-worst degradation separately. A regime label describes information known at the decision time.

In [ ]:
if not regimes.empty:
    display(regimes.sort_values(['model', 'seed', 'regime']))
    regime_summary = regimes.groupby(['model', 'seed'])['sharpe'].agg(mean_regime_sharpe='mean', std_regime_sharpe='std', worst_regime_sharpe='min', best_regime_sharpe='max').reset_index()
    regime_summary['best_minus_worst'] = regime_summary['best_regime_sharpe'] - regime_summary['worst_regime_sharpe']
    display(regime_summary)
    pivot = regimes.groupby(['model', 'regime'])['sharpe'].mean().unstack()
    ax = pivot.plot(kind='bar', figsize=(10, 4), title='Sharpe by decision-time regime'); ax.set_ylabel('Annualized Sharpe'); plt.tight_layout(); plt.show()

## Out-of-sample path and drawdown
The stored returns are gross. The curves below use one observed seed per model; label the seed, dates, and fee convention in any exported figure.

In [ ]:
if not returns.empty:
    first_seeds = returns.groupby('model')['seed'].min().to_dict()
    subset = returns[returns.apply(lambda row: row['seed'] == first_seeds[row['model']], axis=1)].copy()
    subset['target_date'] = pd.to_datetime(subset['target_date'])
    fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
    for model, frame in subset.groupby('model'):
        frame = frame.sort_values('target_date')
        equity = (1 + frame['gross_return']).cumprod()
        label = f'{model} seed {first_seeds[model]}'
        axes[0].plot(frame['target_date'], equity, label=label)
        axes[1].plot(frame['target_date'], equity / equity.cummax() - 1, label=label)
    axes[0].set_title('Observed gross cumulative return'); axes[1].set_title('Observed gross drawdown')
    axes[0].legend(); axes[1].legend(); plt.tight_layout(); plt.show()

## Complexity, turnover, and seed stability
Parameter count and training seconds are measured from the same run. Compare them with net Sharpe and turnover; do not infer deployability.

In [ ]:
if not metrics.empty:
    base = metrics[metrics['cost_bps'] == 5].copy()
    if base.empty: base = metrics.loc[metrics.groupby(['model', 'seed'])['cost_bps'].idxmin()].copy()
    display(base[['model', 'seed', 'parameters', 'train_seconds', 'Annualized Turnover', 'Sharpe Ratio']])
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for model, frame in base.groupby('model'):
        axes[0].scatter(frame['parameters'], frame['Sharpe Ratio'], label=model)
        axes[1].scatter(frame['Annualized Turnover'], frame['Sharpe Ratio'], label=model)
    axes[0].set_xscale('log'); axes[0].set_xlabel('Trainable parameters'); axes[1].set_xlabel('Annualized turnover')
    for ax in axes: ax.set_ylabel('Net Sharpe'); ax.legend()
    plt.tight_layout(); plt.show()
    display(base.groupby('model')['Sharpe Ratio'].agg(['count', 'mean', 'std', 'min', 'max']))

## Interpretation checklist
Confirm actual file hashes and run identity; equal dates and assets; model availability; test independence; transaction-cost treatment; uncertainty intervals; regime sample sizes; and known data, licensing, and survivorship limitations before writing conclusions.

## Persisted summary artifacts
The runner copies Final_Metrics.md from this run directory. The following cell writes only values derived from completed unit CSVs, plus figures and aggregate tables inside this run.

In [ ]:
if not RUN_ID: raise ValueError('RUN_ID is required for the runner report')
if metrics.empty: raise AssertionError('No actual metrics available for Final_Metrics.md')
run_dir = OUTPUT_ROOT / RUN_ID
figure_dir = run_dir / 'figures' / 'summary'
table_dir = run_dir / 'tables' / 'summary'
figure_dir.mkdir(parents=True,exist_ok=True)
table_dir.mkdir(parents=True,exist_ok=True)
summary = metrics.groupby(['model','cost_bps'],as_index=False).agg(
    seeds=('seed','nunique'), folds=('fold','nunique'),
    mean_sharpe=('Sharpe Ratio','mean'), sd_sharpe=('Sharpe Ratio','std'),
    mean_turnover=('Annualized Turnover','mean'),
    mean_directional_accuracy=('directional_accuracy','mean'),
    mean_mse=('mse','mean'))
summary.to_csv(table_dir/'aggregate_metrics.csv',index=False)
if not regimes.empty:
    regime_summary = regimes.groupby(['model','regime','cost_bps'],as_index=False).agg(
        mean_sharpe=('sharpe','mean'),observations=('observations','sum'))
    regime_summary.to_csv(table_dir/'aggregate_regimes.csv',index=False)
else:
    regime_summary = pd.DataFrame()
# Seed/fold rank stability and complexity/compute trade-offs.
base_cost = metrics[metrics['cost_bps']==5].copy()
if base_cost.empty: base_cost = metrics.loc[metrics.groupby(['model','seed','fold'])['cost_bps'].idxmin()].copy()
rank_scores = base_cost.groupby(['seed','fold','model'])['Sharpe Ratio'].mean().unstack('model')
ranks = rank_scores.rank(axis=1,ascending=False,method='average')
ranks.to_csv(table_dir/'rank_stability.csv')
if not ranks.empty:
    ax=ranks.reset_index(drop=True).plot(marker='o',figsize=(10,5),title='Architecture rank stability by seed/fold')
    ax.invert_yaxis(); ax.set(xlabel='Seed/fold unit',ylabel='Rank (1 = best)')
    ax.figure.savefig(figure_dir/'rank_stability.png',dpi=200,bbox_inches='tight'); plt.close(ax.figure)
complexity = base_cost.dropna(subset=['parameters','train_seconds'])
if not complexity.empty:
    fig,ax=plt.subplots(figsize=(8,5))
    for model,frame in complexity.groupby('model'): ax.scatter(frame['parameters'],frame['train_seconds'],label=model,alpha=.8)
    ax.set(xscale='log',xlabel='Trainable parameters',ylabel='Training seconds',title='Complexity versus measured training time'); ax.legend()
    fig.savefig(figure_dir/'parameters_vs_training_time.png',dpi=200,bbox_inches='tight'); plt.close(fig)
# Prediction, signal, residual, and epoch-resource diagnostics.
if not predictions.empty:
    predictions=predictions.copy(); predictions['residual']=predictions['target_return']-predictions['signal']
    fig,axes=plt.subplots(1,3,figsize=(15,4))
    for ax, field, title in ((axes[0], 'signal', 'Signal distributions'), (axes[1], 'residual', 'Prediction residual distributions'), (axes[2], 'position', 'Position/leverage distributions')):
        if field not in predictions: ax.set_visible(False); continue
        for model, frame in predictions.groupby('model'):
            values = pd.to_numeric(frame[field], errors='coerce').dropna().to_numpy()
            if len(values) > 1 and np.unique(values).size > 1: ax.hist(values, bins=min(30, max(5, int(np.sqrt(len(values))))), alpha=.45, density=True, label=str(model))
            elif len(values): ax.axvline(values[0], label=str(model))
        ax.set_title(title); ax.legend()
    fig.tight_layout(); fig.savefig(figure_dir/'signals_and_residuals.png',dpi=200,bbox_inches='tight'); plt.close(fig)
import json
epoch_records=[]
for path in sorted((run_dir/'telemetry').glob('**/events.jsonl')):
    for line in path.read_text(encoding='utf-8').splitlines():
        try:
            row=json.loads(line)
            if row.get('event')=='epoch_complete': epoch_records.append(row)
        except json.JSONDecodeError: pass
epochs=pd.DataFrame(epoch_records)
if not epochs.empty:
    epochs.to_csv(table_dir/'epoch_telemetry.csv',index=False)
    fields=['train_loss','val_loss','learning_rate','gradient_norm','epoch_seconds','throughput_samples_per_second','gpu_memory_used_bytes','gpu_utilization_percent']
    available=[field for field in fields if field in epochs and epochs[field].notna().any()]
    fig,axes=plt.subplots((len(available)+1)//2,2,figsize=(12,3*((len(available)+1)//2)),squeeze=False)
    for ax,field in zip(axes.flat,available):
        for model,frame in epochs.groupby('model'): ax.plot(frame['epoch'],frame[field],alpha=.7,label=model)
        ax.set(title=field,xlabel='Epoch'); ax.legend()
    for ax in axes.flat[len(available):]: ax.remove()
    fig.tight_layout(); fig.savefig(figure_dir/'training_and_resource_telemetry.png',dpi=200,bbox_inches='tight'); plt.close(fig)
cost_plot = summary.pivot(index='cost_bps',columns='model',values='mean_sharpe')
ax = cost_plot.plot(marker='o',figsize=(9,5),title='Observed net Sharpe versus transaction cost')
ax.set(xlabel='Transaction cost (bps)',ylabel='Annualized Sharpe')
ax.figure.savefig(figure_dir/'transaction_cost_sensitivity.png',dpi=200,bbox_inches='tight')
plt.close(ax.figure)
if not returns.empty:
    fig, axes = plt.subplots(2,1,figsize=(10,7),sharex=True)
    for model, frame in returns.groupby('model'):
        seed,fold = frame[['seed','fold']].sort_values(['seed','fold']).iloc[0]
        sample = frame[(frame['seed']==seed)&(frame['fold']==fold)].copy()
        sample['target_date'] = pd.to_datetime(sample['target_date'])
        sample = sample.sort_values('target_date')
        equity = (1+sample['gross_return']).cumprod()
        axes[0].plot(sample['target_date'],equity,label=f'{model} seed {seed} fold {fold}')
        axes[1].plot(sample['target_date'],equity/equity.cummax()-1,label=f'{model} seed {seed} fold {fold}')
    axes[0].set_title('Observed gross cumulative return')
    axes[1].set_title('Observed gross drawdown')
    axes[0].legend(); axes[1].legend()
    fig.tight_layout(); fig.savefig(figure_dir/'equity_and_drawdown.png',dpi=200,bbox_inches='tight'); plt.close(fig)
    crisis_rows=[]; fig,ax=plt.subplots(figsize=(10,5))
    for (model,seed,fold),frame in returns.groupby(['model','seed','fold']):
        frame=frame.copy(); frame['target_date']=pd.to_datetime(frame['target_date']); frame=frame.sort_values('target_date')
        if len(frame)<20: continue
        window=min(63,len(frame)); rolling=(1+frame['gross_return']).rolling(window).apply(np.prod,raw=True)-1
        end=int(rolling.idxmin()) if rolling.notna().any() else int(frame.index[-1]); location=frame.index.get_loc(end)
        crisis=frame.iloc[max(0,location-window+1):location+1]; equity=(1+crisis['gross_return']).cumprod()
        crisis_rows.append({'model':model,'seed':seed,'fold':fold,'start':str(crisis['target_date'].iloc[0].date()),'end':str(crisis['target_date'].iloc[-1].date()),'observations':len(crisis),'cumulative_return':float(equity.iloc[-1]-1),'worst_daily_return':float(crisis['gross_return'].min())})
        ax.plot(np.arange(len(equity)),equity.to_numpy(),alpha=.65,label=f'{model}:{seed}:{fold}')
    pd.DataFrame(crisis_rows).to_csv(table_dir/'crisis_windows.csv',index=False)
    ax.set(title='Worst observed rolling crisis windows',xlabel='Trading days from window start',ylabel='Growth factor')
    if crisis_rows: ax.legend(fontsize=7,ncol=2)
    fig.savefig(figure_dir/'crisis_windows.png',dpi=200,bbox_inches='tight'); plt.close(fig)
if not regime_summary.empty:
    pivot = regime_summary[regime_summary['cost_bps']==5].pivot(index='model',columns='regime',values='mean_sharpe')
    if not pivot.empty:
        ax = pivot.plot(kind='bar',figsize=(11,5),title='Observed net Sharpe by regime, 5 bps')
        ax.set_ylabel('Annualized Sharpe')
        ax.figure.savefig(figure_dir/'regime_sharpe.png',dpi=200,bbox_inches='tight'); plt.close(ax.figure)
five = summary[summary['cost_bps']==5].sort_values('model')
lines = ['# Final Metrics — '+RUN_ID,'',
         'Generated from observed notebook unit outputs. No metrics are imputed.','',
         '## Aggregate economic and predictive metrics','',
         '| Model | Cost bps | Seeds | Folds | Mean Sharpe | SD Sharpe | Mean turnover | Directional accuracy | MSE |',
         '|---|---:|---:|---:|---:|---:|---:|---:|---:|']
for _,row in five.iterrows():
    lines.append(f"| {row['model']} | {row['cost_bps']:.0f} | {row['seeds']:.0f} | {row['folds']:.0f} | {row['mean_sharpe']:.4f} | {row['sd_sharpe']:.4f} | {row['mean_turnover']:.4f} | {row['mean_directional_accuracy']:.4f} | {row['mean_mse']:.6f} |")
lines += ['', '## Provenance', '',
          'Inputs and hashes: metadata/data_manifest.json and metadata/run_manifest.json.',
          'Per-unit metrics, regimes, returns and predictions: tables/<model>/seed_<seed>/fold_<fold>/.',
          'Training events and checkpoints: telemetry/ and checkpoints/.',
          '', '## Interpretation limits', '',
          'These are historical backtests. Review regime sample sizes, costs, signal timing, uncertainty, data quality, macro release timing, and licensing before drawing conclusions.',
          'Unavailable architectures must never be ranked as trained models.', '']
(run_dir/'Final_Metrics.md').write_text('\n'.join(lines),encoding='utf-8')
print('Wrote',run_dir/'Final_Metrics.md')